# NFL Big Data Bowl 2027
## Draft Market Efficiency: Is There Alpha Left in Pre-Draft Athletic Tracking?

**One-sentence claim.** Once draft slot is accounted for with a non-linear expectation curve, neither classic Combine metrics, NGS scores, nor 10 Hz drill tracking explain early-career usage residuals out of sample — evidence that the draft market already prices observable athleticism, which acts as an *eligibility filter*, not a continuous upside gradient.

**What we do.**
1. **Stage 1 — Value over draft (VOD):** estimate \(E[\text{snaps/season} \mid \text{pick}]\) with restricted cubic splines + year effects (cross-fitted). Residual = VOD.
2. **Stage 2 — Market test:** ask whether fixed blocks of pre-draft info (Core Combine, Extended, NGS, Tracking magnitude, Tracking COD) predict VOD out of sample (leave-one-draft-year-out).
3. **Filter vs gradient:** test whether athleticism predicts *being drafted earlier* more than it predicts *beating the pick*.

**What this is not.** Not a talent ranking. Not causal. Snaps measure opportunity, not play quality (EPA/PFF/WAR are unavailable in the public data).

**Data.** Official competition files only: `players`, `combine_results`, `combine_tracking`, `player_career_successes`. Three draft classes (2023–2025).


## 1. Setup


In [ ]:
import os, platform, warnings, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

try:
    from IPython.display import display, Markdown
except Exception:
    def display(x): print(x)
    def Markdown(x): return x

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

SEED = 2027
DT = 0.1
UNDRAFTED_PICK = 300
LAST_SEASON = 2025
MIN_FRAMES = 8
MOVING_SPEED = 1.0
MAX_PLAUSIBLE_SPEED = 13.0
MAX_PLAUSIBLE_ACCEL = 12.0
CV_REPS, CV_K = 15, 5
STAGE1_ALPHA = 1.0
STAGE2_ALPHA = 10.0
N_PERM, N_BOOT = 500, 1000
MIN_N_GROUP = 25
DRILLS = {"FORTY_YARD_DASH": "d40", "THREE_CONE_DRILL": "d3c", "SHORT_SHUTTLE": "dsh"}

print("Run date:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
print("python", platform.python_version(), "| numpy", np.__version__,
      "| pandas", pd.__version__, "| seed", SEED)


## 2. Load competition data and audit tracking


In [ ]:
def find_data_dir():
    env = os.environ.get("BDB_DATA_DIR")
    cands = [Path(env)] if env else []
    cands += [
        Path("/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("/kaggle/input/nfl-big-data-bowl-2027"),
        Path("./data"), Path("."),
    ]
    for p in cands:
        if p and (p / "combine_tracking.csv").exists():
            return p
    root = Path("/kaggle/input")
    if root.exists():
        hits = list(root.rglob("combine_tracking.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("combine_tracking.csv not found. Attach the competition dataset or set BDB_DATA_DIR.")

DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs_bdb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "players": "players.csv",
    "combine_results": "combine_results.csv",
    "combine_tracking": "combine_tracking.csv",
    "career": "player_career_successes.csv",
}
data = {k: pd.read_csv(DATA_DIR / f) for k, f in FILES.items()}
for k, v in data.items():
    print(f"  {FILES[k]:30s} {str(v.shape):>14}")

trk_raw = data["combine_tracking"].copy()
print("\ntracking columns:", list(trk_raw.columns))
print("career columns:", list(data["career"].columns))
print("combine_results columns:", list(data["combine_results"].columns))

if "entity_type" in trk_raw.columns:
    print("\nA) entity_type x draft_year")
    display(trk_raw.groupby(["draft_year", "entity_type"]).size().unstack(fill_value=0))
    trk_raw = trk_raw[trk_raw["entity_type"] == "PLAYER"].copy()
    print("kept PLAYER rows:", len(trk_raw))

print("\nB) Speed sanity by drill x year (flag class-scale issues)")
aud = (trk_raw.groupby(["drill_type", "draft_year"])["s"]
       .agg(rows="count", median_s="median", max_s="max").round(2))
display(aud)


## 3. Position groups (fine when sample allows)

Primary: WR, TE, CB, S, OL, DL if \(n \ge 25\). Fallback: WR/TE, DB, OL, DL.


In [ ]:
def assign_fine(pos):
    if pd.isna(pos):
        return "Other"
    p = str(pos).upper().strip()
    if p == "WR": return "WR"
    if p == "TE": return "TE"
    if p in ("CB", "NB"): return "CB"
    if p in ("FS", "SS", "S", "DB"): return "S"
    if p in ("T", "G", "C", "OT", "OG", "OL"): return "OL"
    if p in ("DE", "DT", "NT", "DL", "EDGE"): return "DL"
    if p in ("LB", "ILB", "OLB", "MLB"): return "LB"
    return "Other"

players = data["players"].copy()
players["pos_fine"] = players["nfl_position"].apply(assign_fine)
print("Fine counts:")
print(players["pos_fine"].value_counts().to_string())

fine_ok = players["pos_fine"].value_counts()
FINE_GROUPS = [g for g in ["WR", "TE", "CB", "S", "OL", "DL"] if fine_ok.get(g, 0) >= MIN_N_GROUP]

if len(FINE_GROUPS) < 3:
    def assign_coarse(pos):
        if pd.isna(pos): return "Other"
        p = str(pos).upper().strip()
        if p in ("WR", "TE"): return "WR/TE"
        if p in ("CB", "FS", "SS", "DB", "S", "NB"): return "DB"
        if p in ("T", "G", "C", "OT", "OG", "OL"): return "OL"
        if p in ("DE", "DT", "NT", "DL", "EDGE"): return "DL"
        return "Other"
    players["position_group"] = players["nfl_position"].apply(assign_coarse)
    GROUPS = [g for g in ["WR/TE", "DB", "OL", "DL"] if (players["position_group"] == g).sum() >= MIN_N_GROUP]
    print("\nUsing COARSE groups:", GROUPS)
else:
    players["position_group"] = players["pos_fine"]
    GROUPS = FINE_GROUPS
    print("\nUsing FINE groups:", GROUPS)


## 4. Drill tracking features (magnitude + change-of-direction)

Three comparable drills only (`FORTY_YARD_DASH`, `THREE_CONE_DRILL`, `SHORT_SHUTTLE`), PLAYER entities, physical-plausibility filter.

**COD features** (shuttle / 3-cone):
- `dir_change_rate_mv` — heading change rate on moving frames
- `speed_loss_on_cut` — mean speed drop on sharp direction changes


In [ ]:
def _wrap(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def event_features(s, a, d, dt=DT, v_min=MOVING_SPEED):
    s = np.asarray(s, float); a = np.asarray(a, float); d = np.asarray(d, float)
    s_v, a_v = s[np.isfinite(s)], a[np.isfinite(a)]
    if len(s_v) < 5 or len(a_v) < 5:
        return None
    f = {
        "max_speed": float(s_v.max()),
        "max_accel": float(a_v.max()),
        "mean_accel": float(a_v.mean()),
        "accel_std": float(a_v.std(ddof=0)),
    }
    f["implausible"] = int((s_v.max() > MAX_PLAUSIBLE_SPEED) or (a_v.max() > MAX_PLAUSIBLE_ACCEL))
    diff = _wrap(np.diff(np.deg2rad(d)))
    mv = np.isfinite(s) & (s >= v_min)
    fin_mv = np.isfinite(diff) & mv[1:] & mv[:-1]
    f["dir_change_rate_mv"] = float(np.abs(diff[fin_mv]).mean() / dt) if fin_mv.sum() >= 6 else np.nan
    if len(s) >= 8 and np.isfinite(diff).sum() >= 6:
        sharp = np.isfinite(diff) & (np.abs(diff) > np.deg2rad(20))
        s0, s1 = s[:-1], s[1:]
        ok = sharp & np.isfinite(s0) & np.isfinite(s1)
        f["speed_loss_on_cut"] = float(np.maximum(0, s0[ok] - s1[ok]).mean()) if ok.sum() >= 3 else np.nan
    else:
        f["speed_loss_on_cut"] = np.nan
    return f

def compute_drill_features(tracking):
    t = tracking[tracking["drill_type"].isin(DRILLS)].copy()
    for c in ["s", "a", "dir"]:
        t[c] = pd.to_numeric(t[c], errors="coerce")
    if "time" not in t.columns:
        t["time"] = np.arange(len(t))
    keys = ["nfl_id", "event_id"]
    t = t.sort_values(keys + ["time"]).reset_index(drop=True)
    S, A, D = t["s"].to_numpy(float), t["a"].to_numpy(float), t["dir"].to_numpy(float)
    DR = t["drill_type"].to_numpy()
    rows, dropped = [], 0
    for (pid, ev), idx in t.groupby(keys, sort=False).indices.items():
        if len(idx) < MIN_FRAMES:
            continue
        f = event_features(S[idx], A[idx], D[idx])
        if f is None:
            continue
        if f.pop("implausible", 0):
            dropped += 1
            continue
        f["nfl_id"] = pid
        f["drill"] = DRILLS.get(DR[idx[0]], "other")
        rows.append(f)
    ev = pd.DataFrame(rows)
    if ev.empty:
        raise RuntimeError("No drill events produced features")
    val_cols = [c for c in ev.columns if c not in ("nfl_id", "drill")]
    wide_parts = []
    for drill, pref in DRILLS.items():
        sub = ev[ev["drill"] == pref]
        if sub.empty:
            continue
        g = sub.groupby("nfl_id")[val_cols].mean().rename(columns={c: f"{pref}_{c}" for c in val_cols})
        wide_parts.append(g)
    player = wide_parts[0]
    for p in wide_parts[1:]:
        player = player.join(p, how="outer")
    player = player.reset_index()
    print(f"Players with drill features: {len(player)} | implausible events dropped: {dropped}")
    return player, ev

movement, event_df = compute_drill_features(trk_raw)
display(movement.head(3))


## 5. Analysis table, volume targets, opportunity-efficiency proxy

- **Primary Y:** snaps per season of exposure (removes cohort bias of career totals).
- **Secondary:** `starts_ps`, `start_rate` = games_started / games_active (opportunity efficiency proxy).
- EPA / PFF / WAR are **not** in the public file; pipeline is ready to swap residual target when available.


In [ ]:
def build_table(players, combine_results, movement, career):
    df = players.merge(combine_results, on="nfl_id", how="left", suffixes=("", "_cr"))
    df = df.merge(movement, on="nfl_id", how="left")
    df = df.merge(career, on="nfl_id", how="left", suffixes=("", "_car"))
    df["draft_capital"] = pd.to_numeric(df.get("draft_overall_pick"), errors="coerce").fillna(UNDRAFTED_PICK)
    df["undrafted"] = df.get("draft_overall_pick").isna().astype(float)
    df["draft_year"] = pd.to_numeric(df["draft_year"], errors="coerce")
    df["seasons_exposed"] = (LAST_SEASON - df["draft_year"] + 1).clip(lower=1)
    for t in ["career_offensive_snaps", "career_defensive_snaps"]:
        if t in df.columns:
            df[t + "_ps"] = pd.to_numeric(df[t], errors="coerce") / df["seasons_exposed"]
    if "career_games_started" in df.columns:
        df["starts_ps"] = pd.to_numeric(df["career_games_started"], errors="coerce") / df["seasons_exposed"]
    else:
        df["starts_ps"] = np.nan
    # FIXED: use career_games_active from the actual schema
    if "career_games_started" in df.columns and "career_games_active" in df.columns:
        gs = pd.to_numeric(df["career_games_started"], errors="coerce")
        ga = pd.to_numeric(df["career_games_active"], errors="coerce").replace(0, np.nan)
        df["start_rate"] = gs / ga
    else:
        df["start_rate"] = np.nan
    return df

analysis = build_table(players, data["combine_results"], movement, data["career"])
print("Analysis:", analysis.shape)
print("Draft years:", sorted(analysis["draft_year"].dropna().unique().astype(int).tolist()))
print("Undrafted:", int(analysis["undrafted"].sum()))

OFFENSE = {"WR", "TE", "OL", "WR/TE"}
TARGET_SNAPS = {g: ("career_offensive_snaps_ps" if g in OFFENSE else "career_defensive_snaps_ps") for g in GROUPS}
print("Snap targets:", TARGET_SNAPS)
print("start_rate non-null:", int(analysis["start_rate"].notna().sum()))
print("starts_ps non-null:", int(analysis["starts_ps"].notna().sum()))
print("NGS cols present:", [c for c in analysis.columns if c.startswith("ngs_")])


## 6. Stage 1 — Non-linear \(E[\text{snaps} \mid \text{pick}]\) and VOD

Spline basis on draft capital + undrafted flag + draft-year dummies. Repeated K-fold cross-fitting so no player is scored by a model that saw him.


In [ ]:
def crossfit_expected(df, ycol, pick_col="draft_capital", n_reps=CV_REPS, k=CV_K, seed=SEED):
    sub = df.dropna(subset=[ycol, pick_col]).copy()
    y = sub[ycol].to_numpy(float)
    pick = sub[pick_col].to_numpy(float).reshape(-1, 1)
    und = sub["undrafted"].to_numpy(float).reshape(-1, 1) if "undrafted" in sub.columns else np.zeros((len(sub), 1))
    years = sorted(sub["draft_year"].dropna().unique())
    year_mat = (np.column_stack([(sub["draft_year"] == yr).astype(float).to_numpy() for yr in years[:-1]])
                if len(years) > 1 else np.zeros((len(sub), 0)))
    acc = np.zeros(len(sub)); cnt = np.zeros(len(sub))
    rng = np.random.default_rng(seed)
    for rep in range(n_reps):
        kf = KFold(n_splits=k, shuffle=True, random_state=int(rng.integers(0, 1_000_000)))
        fold_pred = np.full(len(sub), np.nan)
        for tr, te in kf.split(pick):
            spl = SplineTransformer(n_knots=4, degree=3, include_bias=False)
            Xtr = np.hstack([spl.fit_transform(pick[tr]), und[tr], year_mat[tr]])
            Xte = np.hstack([spl.transform(pick[te]), und[te], year_mat[te]])
            med = np.nanmedian(Xtr, axis=0)
            Xtr = np.where(np.isfinite(Xtr), Xtr, med)
            Xte = np.where(np.isfinite(Xte), Xte, med)
            model = Ridge(alpha=STAGE1_ALPHA, random_state=SEED)
            model.fit(Xtr, y[tr])
            fold_pred[te] = model.predict(Xte)
        m = np.isfinite(fold_pred)
        acc[m] += fold_pred[m]; cnt[m] += 1
    exp = np.where(cnt > 0, acc / cnt, np.nan)
    out = sub[["nfl_id"]].copy()
    out["exp"] = exp; out["y"] = y; out["vod"] = y - exp
    return out

stage1_rows = []
analysis = analysis.copy()
for col in ("exp_snaps", "vod_snaps", "vod_z_snaps"):
    analysis[col] = np.nan

for g in GROUPS:
    ycol = TARGET_SNAPS[g]
    mask = analysis["position_group"] == g
    sub = analysis.loc[mask]
    if sub[ycol].notna().sum() < MIN_N_GROUP:
        print(f"skip {g}: n too small"); continue
    cf = crossfit_expected(sub, ycol)
    tmp = analysis.loc[mask, ["nfl_id"]].merge(cf, on="nfl_id", how="left")
    analysis.loc[mask, "exp_snaps"] = tmp["exp"].values
    analysis.loc[mask, "vod_snaps"] = tmp["vod"].values
    sd = float(np.nanstd(tmp["vod"].values))
    analysis.loc[mask, "vod_z_snaps"] = tmp["vod"].values / sd if sd > 0 else np.nan
    valid = np.isfinite(tmp["exp"].values) & np.isfinite(tmp["y"].values)
    r2v = r2_score(tmp["y"].values[valid], tmp["exp"].values[valid]) if valid.sum() > 10 else np.nan
    stage1_rows.append({"group": g, "n": int(np.isfinite(tmp["vod"]).sum()),
                        "R2_draft_OOF": round(float(r2v), 3), "SD_VOD": round(sd, 1)})

stage1 = pd.DataFrame(stage1_rows)
print("Stage 1 — how much usage draft slot explains (nonlinear, OOF):")
display(stage1)


## 7. Stage 2 — Do pre-draft blocks explain VOD? (efficient-market test)

**Fixed blocks** (same for every position; no post-hoc feature selection):

| Block | Contents |
|---|---|
| Core Combine | forty, short_shuttle, vertical |
| Extended Combine | + broad, 3-cone, split, bench, anthropometrics |
| **NGS scores** | ngs_athleticism_score, ngs_college_production_score, ngs_final_score |
| Tracking magnitude | max speed / accel by drill |
| Tracking COD | dir_change_rate_mv, speed_loss_on_cut |
| All athletic | union of the above |

Validation: leave-one-draft-year-out. Uncertainty: bootstrap CI + within-year permutation + Holm across all tests.


In [ ]:
def existing(cols, df):
    return [c for c in cols if c in df.columns]

CORE = existing(["forty", "short_shuttle", "vertical"], analysis)
EXTENDED = existing(
    ["forty", "short_shuttle", "vertical", "broad_jump", "three_cone", "ten_yd_split",
     "bench_reps", "combine_height", "combine_weight", "arm_length", "hand_size", "wing_span"], analysis)
NGS = existing(["ngs_athleticism_score", "ngs_college_production_score", "ngs_final_score"], analysis)
TRACK_MAG = existing([f"{p}_{m}" for p in ("d40", "d3c", "dsh") for m in ("max_speed", "max_accel")], analysis)
TRACK_COD = existing([f"{p}_{m}" for p in ("d3c", "dsh") for m in ("dir_change_rate_mv", "speed_loss_on_cut")], analysis)
ALL_ATH = list(dict.fromkeys(CORE + EXTENDED + NGS + TRACK_MAG + TRACK_COD))

BLOCKS = {
    "Core Combine": CORE,
    "Extended Combine": EXTENDED,
    "NGS scores": NGS,
    "Tracking magnitude": TRACK_MAG,
    "Tracking COD": TRACK_COD,
    "All athletic": ALL_ATH,
}
for name, cols in BLOCKS.items():
    print(f"  {name:22s} ({len(cols):2d} feats)")

def oof_logo(df, features, target, group_col="draft_year"):
    sub = df.dropna(subset=[target]).copy()
    if len(sub) < MIN_N_GROUP or not features:
        return None, None, None
    X, y = sub[features], sub[target].to_numpy(float)
    groups = sub[group_col].to_numpy()
    if len(np.unique(groups)) < 2:
        return None, None, None
    oof = np.full(len(sub), np.nan)
    for tr, te in LeaveOneGroupOut().split(X, y, groups):
        pipe = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("m", Ridge(alpha=STAGE2_ALPHA, random_state=SEED)),
        ])
        pipe.fit(X.iloc[tr], y[tr])
        oof[te] = pipe.predict(X.iloc[te])
    return y, oof, groups

def bootstrap_r2(y, pred, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    stats = []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if np.isfinite(pred[idx]).sum() < 10:
            continue
        stats.append(r2_score(y[idx], pred[idx]))
    if not stats:
        return np.nan, np.nan, np.nan
    return float(np.mean(stats)), float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))

def perm_r2(y, pred, groups, n_perm=N_PERM, seed=SEED):
    rng = np.random.default_rng(seed)
    obs = r2_score(y, pred)
    count = 0
    for _ in range(n_perm):
        perm = pred.copy()
        for g in np.unique(groups):
            m = groups == g
            perm[m] = rng.permutation(perm[m])
        if r2_score(y, perm) >= obs:
            count += 1
    return (count + 1) / (n_perm + 1)

def holm(pvals):
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m); running = 0.0
    for i, idx in enumerate(order):
        val = min(1.0, (m - i) * pvals[idx])
        running = max(running, val)
        adj[idx] = running
    return adj

stage2_rows = []
for g in GROUPS:
    sub = analysis[(analysis["position_group"] == g) & analysis["vod_snaps"].notna()].copy()
    if len(sub) < MIN_N_GROUP:
        continue
    for bname, feats in BLOCKS.items():
        feats = existing(feats, sub)
        if len(feats) < 1:
            continue
        y, oof, groups = oof_logo(sub, feats, "vod_snaps")
        if y is None:
            continue
        r2v = r2_score(y, oof)
        r2m, lo, hi = bootstrap_r2(y, oof)
        p = perm_r2(y, oof, groups)
        stage2_rows.append({"group": g, "block": bname, "n": len(y), "n_feat": len(feats),
                            "r2": r2v, "r2_boot": r2m, "r2_lo": lo, "r2_hi": hi, "p_perm": p})

stage2 = pd.DataFrame(stage2_rows)
if len(stage2):
    stage2["p_holm"] = holm(stage2["p_perm"].to_numpy())
    stage2["verdict"] = "No gain"
    stage2.loc[(stage2["r2_lo"] > 0) & (stage2["p_holm"] <= 0.10), "verdict"] = "Supported"
    stage2.loc[(stage2["r2"] > 0) & (stage2["verdict"] != "Supported"), "verdict"] = "Suggestive"
    display(stage2.round(3))
else:
    print("Stage 2 empty — check features / n")


## 8. Filter vs gradient

Athleticism score = PCA-1 of Core Combine (times inverted so higher = better).

- **Filter:** association with draft capital (does it move the pick?).
- **Gradient:** association with VOD (does extra athleticism beat the pick?).


In [ ]:
core_cols = existing(["forty", "short_shuttle", "vertical"], analysis)
score_df = analysis.dropna(subset=core_cols + ["vod_snaps", "draft_capital"]).copy()
filter_vs_gradient = {}

if len(core_cols) >= 2 and len(score_df) > 50:
    Z = score_df[core_cols].to_numpy(float)
    for i, c in enumerate(core_cols):
        if c in ("forty", "short_shuttle", "three_cone", "ten_yd_split"):
            Z[:, i] = -Z[:, i]
    Z = (Z - np.nanmean(Z, axis=0)) / (np.nanstd(Z, axis=0) + 1e-9)
    Z = np.where(np.isfinite(Z), Z, 0)
    score_df["ath_score"] = PCA(n_components=1, random_state=SEED).fit_transform(Z).ravel()

    corr_pick = float(np.corrcoef(score_df["ath_score"], score_df["draft_capital"])[0, 1])
    corr_vod = float(np.corrcoef(score_df["ath_score"], score_df["vod_snaps"])[0, 1])
    print(f"corr(ath_score, draft_capital) = {corr_pick:+.3f}   <-- filter / market")
    print(f"corr(ath_score, VOD_snaps)     = {corr_vod:+.3f}   <-- residual gradient")

    score_df["ath_tercile"] = pd.qcut(score_df["ath_score"], 3, labels=["low", "mid", "high"])
    tab = score_df.groupby("ath_tercile", observed=True).agg(
        n=("nfl_id", "count"),
        mean_pick=("draft_capital", "mean"),
        mean_vod=("vod_snaps", "mean"),
    ).round(2)
    print("\nAthleticism terciles:")
    display(tab)

    if abs(corr_pick) > abs(corr_vod) + 0.05:
        interp = "Filter-dominant: athleticism tracks the pick, not residual VOD"
    elif abs(corr_vod) > 0.15:
        interp = "Residual gradient possible"
    else:
        interp = "Both weak in this sample"
    filter_vs_gradient = {"corr_ath_vs_pick": corr_pick, "corr_ath_vs_vod": corr_vod,
                         "interpretation": interp, "terciles": tab}
    print("Interpretation:", interp)
else:
    filter_vs_gradient = {"interpretation": "insufficient data"}
    print(filter_vs_gradient)


## 9. Figures for the write-up


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))

# --- Panel A: Stage 1 expected vs actual ---
ax = axes[0]
m = analysis["exp_snaps"].notna() & analysis["vod_snaps"].notna()
actual = analysis.loc[m, "exp_snaps"] + analysis.loc[m, "vod_snaps"]
ax.scatter(analysis.loc[m, "exp_snaps"], actual, alpha=0.30, s=14, c="#1f77b4", edgecolors="none")
lim = max(float(analysis.loc[m, "exp_snaps"].max()), float(actual.max())) * 1.05
ax.plot([0, lim], [0, lim], "k--", lw=1)
ax.set_xlabel("E[snaps/season | pick] (spline, OOF)")
ax.set_ylabel("Actual snaps/season")
ax.set_title("A. Stage 1: draft expectation vs actual")

# --- Panel B: Stage 2 R2 heatmap ---
ax = axes[1]
if len(stage2):
    piv = stage2.pivot_table(index="block", columns="group", values="r2", aggfunc="mean")
    im = ax.imshow(piv.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-0.2, vmax=0.2)
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(list(piv.columns), rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(list(piv.index), fontsize=8)
    ax.set_title("B. Stage 2: OOS R² (block → VOD)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
else:
    ax.text(0.5, 0.5, "no stage2", ha="center"); ax.set_axis_off()

# --- Panel C: filter vs gradient terciles ---
ax = axes[2]
if "terciles" in filter_vs_gradient:
    t = filter_vs_gradient["terciles"].reset_index()
    x = np.arange(len(t))
    w = 0.35
    ax.bar(x - w/2, t["mean_pick"], w, label="mean pick", color="#4c72b0")
    ax2 = ax.twinx()
    ax2.bar(x + w/2, t["mean_vod"], w, label="mean VOD", color="#dd8452")
    ax.set_xticks(x); ax.set_xticklabels(t["ath_tercile"])
    ax.set_ylabel("Mean draft capital (lower = earlier)")
    ax2.set_ylabel("Mean VOD (snaps/season)")
    ax.set_title("C. Filter vs gradient by ath tercile")
    ax.legend(loc="upper left", fontsize=8)
    ax2.legend(loc="upper right", fontsize=8)
else:
    ax.text(0.5, 0.5, "no terciles", ha="center"); ax.set_axis_off()

plt.tight_layout()
fig.savefig(OUT_DIR / "fig_main.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", OUT_DIR / "fig_main.png")


## 10. Auto-generated summary (paste into write-up)


In [ ]:
def summary_md():
    L = []
    L.append("# Draft market efficiency — results summary\n")
    L.append("## Claim")
    L.append(
        "After a non-linear draft-slot expectation is removed, no fixed block of pre-draft "
        "athletic information (Combine, NGS, or 10 Hz drill tracking) explains early-career "
        "usage residuals out of sample. Athleticism behaves as an **eligibility filter**, "
        "not a continuous value-over-pick gradient.\n"
    )
    if len(stage1):
        L.append("## Stage 1 — draft slot already prices usage")
        L.append("| Group | N | R² OOF | SD(VOD) |")
        L.append("|---|---:|---:|---:|")
        for _, r in stage1.iterrows():
            L.append(f"| {r['group']} | {r['n']} | {r['R2_draft_OOF']:.2f} | {r['SD_VOD']:.0f} |")
        L.append("")
    if len(stage2):
        sup = stage2[stage2["verdict"] == "Supported"]
        sug = stage2[stage2["verdict"] == "Suggestive"]
        L.append("## Stage 2 — efficient-market test")
        if len(sup) == 0:
            L.append(
                "**No Supported block.** No pre-draft athletic block has a bootstrap CI for R² "
                "that excludes zero after Holm adjustment."
            )
        else:
            L.append("Supported: " + "; ".join(f"{r['group']}/{r['block']} R²={r['r2']:+.2f}" for _, r in sup.iterrows()))
        if len(sug):
            L.append(
                "Suggestive only (not Supported): "
                + "; ".join(f"{r['group']}/{r['block']} R²={r['r2']:+.2f}" for _, r in sug.iterrows())
            )
        neg = stage2[stage2["r2"] < 0]
        L.append(
            f"Blocks with negative OOS R² (overfit / no signal): {len(neg)} / {len(stage2)}. "
            "High-dimensional blocks (All athletic, Tracking magnitude) are systematically negative.\n"
        )
    L.append("## Filter vs gradient")
    L.append(filter_vs_gradient.get("interpretation", "n/a"))
    if "corr_ath_vs_pick" in filter_vs_gradient:
        L.append(f"- corr(athleticism, pick) = {filter_vs_gradient['corr_ath_vs_pick']:+.3f}")
        L.append(f"- corr(athleticism, VOD)  = {filter_vs_gradient['corr_ath_vs_vod']:+.3f}\n")
    L.append("## Implications for front offices and the league")
    L.append("1. Treat 1v0 drill athleticism as a **threshold for eligibility**, not a ranking of residual upside.")
    L.append("2. Marginal value of more resolution on isolated drills is limited once the pick is known.")
    L.append("3. Shift tracking investment toward **college game film (11v11)**, where context is not already absorbed by the draft market.")
    L.append("4. If private EPA / PFF / WAR exist, swap them as Stage-1 targets; the rest of the pipeline is unchanged.\n")
    L.append("## Caveats")
    L.append("- Three draft classes; leave-one-year-out; Holm over all Stage-2 tests.")
    L.append("- `LAST_SEASON=2025` and undrafted=pick 300 are assumptions.")
    L.append("- Snaps measure opportunity (scheme, depth, investment), not play quality.")
    L.append("- Tracking limited to three comparable drills with a plausibility filter.")
    L.append("- Predictive, not causal; leaderboards (if built) are descriptive only.")
    return "\n".join(L)

md_text = summary_md()
display(Markdown(md_text))

stage1.to_csv(OUT_DIR / "stage1_skill.csv", index=False)
if len(stage2):
    stage2.to_csv(OUT_DIR / "stage2_market_test.csv", index=False)
keep = [c for c in [
    "nfl_id", "display_name", "position_group", "nfl_position", "draft_year",
    "draft_overall_pick", "draft_capital", "seasons_exposed",
    "exp_snaps", "vod_snaps", "vod_z_snaps", "starts_ps", "start_rate",
] if c in analysis.columns]
analysis[keep].to_csv(OUT_DIR / "players_vod.csv", index=False)
(OUT_DIR / "writeup_summary.md").write_text(md_text, encoding="utf-8")
print("\nExported to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob("*")):
    if p.suffix in (".csv", ".md", ".png"):
        print(" ", p.name)


## 11. Suggested write-up structure (competition submission)

1. **Introduction** — Research question: *Is there undiscovered alpha in pre-draft drill tracking after the market has set a pick?*
2. **Data** — Official BDB files; PLAYER-only tracking; three drills; plausibility filter; fine position groups.
3. **Methods**
   - Stage 1: spline \(E[\text{snaps/season} \mid \text{pick}, \text{year}]\) → VOD (cross-fitted).
   - Stage 2: fixed feature blocks → VOD, leave-one-draft-year-out, bootstrap + Holm.
   - Filter vs gradient: PCA athleticism score vs pick and vs VOD.
4. **Results** — Stage-1 R² table; Stage-2 heatmap (mostly ≤ 0); filter-dominant correlations; Figure 1 (three panels).
5. **Discussion** — Efficient draft market for observable athleticism; drills as eligibility filter; recommend 11v11 college tracking.
6. **Limitations** — Volume ≠ quality; short careers; three classes; no EPA/PFF in public data.
7. **Appendix** — Feature definitions, audit table, full Stage-2 CSV.

**Headline you can defend:**  
*"Draft slot explains 30–70% of early usage. Nothing in the Combine, NGS, or 10 Hz drill tracking explains the residual out of sample. Athleticism is a filter, not a gradient — the market already priced it."*
